# 1. Introduction and research question

We study whether oil price increases have different effects on financial markets depending on whether they are more closely associated with demand conditions or with a residual non-demand-related component.

The notebook keeps the empirical strategy deliberately simple. The main analysis is monthly, the oil decomposition is reduced-form, and the predictive regressions are interpreted as forecasting relationships rather than full causal identification.

# 2. Imports and setup

In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, grangercausalitytests, acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats
from scipy.optimize import minimize
from scipy.stats import chi2 as chi2_dist

warnings.filterwarnings("ignore")
plt.style.use("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:.4f}".format)

ModuleNotFoundError: No module named 'matplotlib'

# 3. Constants and helper functions

All analysis functions are defined below — no external imports from `src/`.

## 3a. Constants

In [ ]:
DAILY_COLUMNS = [
    "date", "wti", "brent", "sp500", "msci_em",
    "us10y", "us2y", "hy_ytw", "gold",
]

MONTHLY_COLUMNS = ["date", "cfnai", "ism_mfg"]

SHEET_USECOLS = {
    "Daily": [0, 1, 2, 6, 8, 10, 11, 12, 13],
    "Monthly": [0, 2, 3],
}

## 3b. Data loading

In [ ]:
def load_data_sheet(excel_path, sheet_name, columns):
    raw = pd.read_excel(
        excel_path, sheet_name=sheet_name, header=None,
        usecols=SHEET_USECOLS[sheet_name],
    )
    data = raw.iloc[5:].copy()
    data.columns = columns

    first_value = str(data.iloc[0, 0]).strip().lower()
    if first_value == "dates":
        data = data.iloc[1:].copy()

    data["date"] = pd.to_datetime(data["date"])
    for column in columns[1:]:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    data = data.sort_values("date").reset_index(drop=True)
    return data

In [ ]:
def aggregate_daily_to_monthly(daily_df):
    monthly_data = daily_df.resample("M", on="date").last().reset_index()

    sp500_daily_returns = daily_df[["date", "sp500"]].copy()
    sp500_daily_returns["sp500_daily_log_return"] = np.log(sp500_daily_returns["sp500"]).diff()

    monthly_volatility = (
        sp500_daily_returns.resample("M", on="date")["sp500_daily_log_return"]
        .std()
        .rename("sp500_realized_vol")
        .reset_index()
    )

    monthly_data = monthly_data.merge(monthly_volatility, on="date", how="left")
    return monthly_data

## 3c. Variable construction

In [ ]:
def add_log_return(df, price_column, new_column):
    df[new_column] = np.log(df[price_column] / df[price_column].shift(1))

In [ ]:
def build_project_variables(monthly_df):
    df = monthly_df.copy()
    df = df.sort_values("date").reset_index(drop=True)

    add_log_return(df, "wti", "wti_return")
    add_log_return(df, "brent", "brent_return")
    add_log_return(df, "sp500", "sp500_return")
    add_log_return(df, "msci_em", "msci_em_return")
    add_log_return(df, "gold", "gold_return")

    df["term_spread"] = df["us10y"] - df["us2y"]
    df["hy_change"] = df["hy_ytw"].diff()

    print("\nCheck of expected columns after variable construction:")
    expected_columns = [
        "wti_return", "brent_return", "sp500_return", "msci_em_return",
        "gold_return", "term_spread", "hy_change", "sp500_realized_vol",
        "cfnai", "ism_mfg",
    ]
    for column in expected_columns:
        print(f"{column}: {'OK' if column in df.columns else 'MISSING'}")

    return df

## 3d. ADF stationarity tests

In [ ]:
def run_adf_table(df, columns):
    rows = []
    for column in columns:
        series = df[column].dropna()
        if len(series) < 20:
            rows.append({"variable": column, "n_obs": len(series), "adf_stat": np.nan, "p_value": np.nan, "stationary_5pct": np.nan})
            continue
        result = adfuller(series, autolag="AIC")
        rows.append({"variable": column, "n_obs": len(series), "adf_stat": round(result[0], 4), "p_value": round(result[1], 4), "stationary_5pct": result[1] < 0.05})
    return pd.DataFrame(rows)

## 3e. Ljung-Box and Jarque-Bera diagnostics

In [ ]:
def run_ljungbox_jb_table(df, columns, lags=(6, 12)):
    rows = []
    for col in columns:
        s = df[col].dropna()
        if len(s) < 20:
            continue
        lb = acorr_ljungbox(s, lags=list(lags), return_df=True)
        jb_stat, jb_p = stats.jarque_bera(s.values)
        row = {"variable": col}
        for lag in lags:
            row[f"LB({lag})_stat"] = round(float(lb.loc[lag, "lb_stat"]), 2)
            row[f"LB({lag})_pval"] = round(float(lb.loc[lag, "lb_pvalue"]), 4)
        row["JB_stat"] = round(float(jb_stat), 2)
        row["JB_pval"] = float(jb_p)
        row["normal_5pct"] = jb_p > 0.05
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
def run_diagnostic_residuals(resid, lags=(6, 12)):
    resid = np.asarray(resid).flatten()
    resid = resid[np.isfinite(resid)]
    lb = acorr_ljungbox(resid, lags=list(lags), return_df=True)
    acf_values = acf(resid, nlags=max(lags), fft=True)
    rows = []
    for lag in lags:
        rows.append({"lag": lag, "ACF": round(float(acf_values[lag]), 4), "LB_stat": round(float(lb.loc[lag, "lb_stat"]), 2), "LB_pval": round(float(lb.loc[lag, "lb_pvalue"]), 4), "autocorrelated_5pct": lb.loc[lag, "lb_pvalue"] < 0.05})
    return pd.DataFrame(rows)

## 3f. OLS from scratch

In [ ]:
def ols_from_scratch(y, X, x_names=None):
    mask = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
    y, X = y[mask], X[mask]
    T = len(y)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    k = X.shape[1]
    Xc = np.column_stack([np.ones(T), X])
    XtX_inv = np.linalg.inv(Xc.T @ Xc)
    beta = XtX_inv @ (Xc.T @ y)
    y_hat = Xc @ beta
    resid = y - y_hat
    SSR = resid @ resid
    SST = (y - y.mean()) @ (y - y.mean())
    R2 = 1.0 - SSR / SST
    sigma2 = SSR / (T - k - 1)
    var_beta = sigma2 * XtX_inv
    se = np.sqrt(np.diag(var_beta))
    t_stat = beta / se
    p_value = 2.0 * (1.0 - stats.t.cdf(np.abs(t_stat), df=T - k - 1))
    names = ["const"] + (x_names if x_names else [f"x{i}" for i in range(k)])
    return {"beta": beta, "se": se, "t_stat": t_stat, "p_value": p_value, "resid": resid, "fitted": y_hat, "R2": R2, "sigma2": sigma2, "T": T, "k": k + 1, "names": names, "y": y, "X": Xc}

In [ ]:
def ols_summary_df(res):
    return pd.DataFrame({"coef": res["beta"], "std_err": res["se"], "t_stat": res["t_stat"], "p_value": res["p_value"]}, index=res["names"]).round(4)

## 3g. MA(1) MLE with recursive innovations

In [ ]:
def ma_innovations(u, theta):
    T = len(u)
    eps = np.zeros(T)
    eps[0] = u[0]
    for t in range(1, T):
        eps[t] = u[t] - theta * eps[t - 1]
    return eps

In [ ]:
def _loglik_contributions(params, y, X):
    k = X.shape[1]
    beta = params[:k]
    theta = params[k]
    u = y - X @ beta
    eps = ma_innovations(u, theta)
    sigma2 = np.mean(eps ** 2)
    if sigma2 <= 0:
        return np.full(len(y), -1e10)
    return -0.5 * np.log(2 * np.pi * sigma2) - 0.5 * eps ** 2 / sigma2

def _negloglik_concentrated(params, y, X):
    ll = _loglik_contributions(params, y, X)
    return -np.sum(ll)

In [ ]:
def opg_standard_errors(params, y, X):
    T = len(y)
    n_params = len(params)
    scores = np.zeros((T, n_params))
    h = 1e-5
    for j in range(n_params):
        p_up = params.copy(); p_up[j] += h
        p_dn = params.copy(); p_dn[j] -= h
        scores[:, j] = (_loglik_contributions(p_up, y, X) - _loglik_contributions(p_dn, y, X)) / (2 * h)
    OPG = scores.T @ scores
    try:
        se = np.sqrt(np.diag(np.linalg.inv(OPG)))
    except np.linalg.LinAlgError:
        se = np.full(n_params, np.nan)
    return se

In [ ]:
def estimate_ma1(y, X, x_names=None):
    ols_res = ols_from_scratch(y, X, x_names)
    y_clean, X_clean = ols_res["y"], ols_res["X"]
    x0 = np.concatenate([ols_res["beta"], [0.0]])
    result = minimize(_negloglik_concentrated, x0, args=(y_clean, X_clean), method="Nelder-Mead", options={"maxiter": 10000, "xatol": 1e-8, "fatol": 1e-10})
    params = result.x
    se = opg_standard_errors(params, y_clean, X_clean)
    k = X_clean.shape[1]
    beta = params[:k]
    theta = params[k]
    u = y_clean - X_clean @ beta
    innovations = ma_innovations(u, theta)
    sigma = np.sqrt(np.mean(innovations ** 2))
    names = ols_res["names"] + ["theta_MA1"]
    return {"beta": beta, "theta": theta, "se": se, "sigma": sigma, "innovations": innovations, "residuals_ols": ols_res["resid"], "params": params, "names": names, "ols_beta": ols_res["beta"], "ols_se": ols_res["se"], "T": ols_res["T"], "R2_ols": ols_res["R2"]}

In [ ]:
def ma1_comparison_table(ma1_res):
    names = ma1_res["names"]
    n_beta = len(ma1_res["ols_beta"])
    rows = []
    for i in range(n_beta):
        rows.append({"parameter": names[i], "OLS_coef": round(float(ma1_res["ols_beta"][i]), 4), "OLS_se": round(float(ma1_res["ols_se"][i]), 4), "MLE_coef": round(float(ma1_res["beta"][i]), 4), "MLE_se": round(float(ma1_res["se"][i]), 4)})
    rows.append({"parameter": "theta_MA1", "OLS_coef": np.nan, "OLS_se": np.nan, "MLE_coef": round(float(ma1_res["theta"]), 4), "MLE_se": round(float(ma1_res["se"][-1]), 4)})
    return pd.DataFrame(rows)

## 3h. Oil shock decomposition

In [ ]:
def decompose_oil_returns(df, oil_return_col="wti_return", activity_col="cfnai", prefix="baseline"):
    sample = df[[oil_return_col, activity_col]].dropna().copy()
    X = sm.add_constant(sample[activity_col])
    model = sm.OLS(sample[oil_return_col], X).fit()
    fitted = pd.Series(index=sample.index, data=model.fittedvalues, name=f"{prefix}_oil_demand_component")
    residual = pd.Series(index=sample.index, data=model.resid, name=f"{prefix}_oil_supply_component")
    out = df.copy()
    out[fitted.name] = fitted
    out[residual.name] = residual
    return out, model

In [ ]:
def decompose_oil_returns_scratch(df, oil_return_col="wti_return", activity_col="cfnai", prefix="baseline"):
    sample = df[[oil_return_col, activity_col]].dropna().copy()
    y = sample[oil_return_col].values
    X = sample[activity_col].values.reshape(-1, 1)
    res = ols_from_scratch(y, X, [activity_col])
    out = df.copy()
    out[f"{prefix}_oil_demand_component"] = np.nan
    out[f"{prefix}_oil_supply_component"] = np.nan
    out.loc[sample.index, f"{prefix}_oil_demand_component"] = res["fitted"]
    out.loc[sample.index, f"{prefix}_oil_supply_component"] = res["resid"]
    return out, res

In [ ]:
def add_regime_variables(df, oil_col="wti_return", ism_col="ism_mfg"):
    out = df.copy()
    out["expansion"] = np.where(out[ism_col] > 50, 1, 0)
    out["contraction"] = np.where(out[ism_col] <= 50, 1, 0)
    out["oil_expansion"] = out[oil_col] * out["expansion"]
    out["oil_contraction"] = out[oil_col] * out["contraction"]
    return out

## 3i. Predictive regressions

In [ ]:
def fit_predictive_regression(df, dependent_col, predictor_cols, horizon=1, cov_type="HC1"):
    work = df.copy()
    work["y_next"] = work[dependent_col].shift(-horizon)
    if dependent_col in predictor_cols:
        work = work.rename(columns={dependent_col: f"{dependent_col}_current"})
        predictor_cols = [f"{dependent_col}_current" if col == dependent_col else col for col in predictor_cols]
    regression_columns = ["y_next"] + predictor_cols
    sample = work[regression_columns].dropna().copy()
    y = sample["y_next"]
    X = sm.add_constant(sample[predictor_cols])
    model = sm.OLS(y, X).fit(cov_type=cov_type)
    return model, sample

In [ ]:
def regression_results_table(model):
    table = pd.DataFrame({"coef": model.params, "std_err": model.bse, "t_stat": model.tvalues, "p_value": model.pvalues})
    return table.round(4)

In [ ]:
def interpret_two_component_model(model, demand_name, supply_name):
    lines = []
    for variable, label in [(demand_name, "Demand-related oil component"), (supply_name, "Supply-related oil component")]:
        if variable not in model.params.index:
            continue
        coef = model.params[variable]
        p_value = model.pvalues[variable]
        sign = "positive" if coef > 0 else "negative"
        strength = "statistically significant" if p_value < 0.05 else "not statistically significant"
        lines.append(f"{label}: coefficient is {sign} ({coef:.4f}) and {strength} at the 5% level (p-value = {p_value:.4f}).")
    return lines

## 3j. Rolling predictive coefficients

In [ ]:
def rolling_predictive_coefficients(df, dependent_col, predictor_cols, window=60, horizon=1):
    work = df.copy()
    work["y_next"] = work[dependent_col].shift(-horizon)
    cols_needed = ["date", "y_next"] + predictor_cols
    work = work[cols_needed].dropna().reset_index(drop=True)
    results = []
    for i in range(window, len(work)):
        train = work.iloc[i - window:i]
        y = train["y_next"].values
        X = sm.add_constant(train[predictor_cols].values)
        try:
            XtX_inv = np.linalg.inv(X.T @ X)
            beta = XtX_inv @ (X.T @ y)
            resid = y - X @ beta
            sigma2 = (resid @ resid) / (len(y) - X.shape[1])
            se = np.sqrt(np.diag(sigma2 * XtX_inv))
        except np.linalg.LinAlgError:
            continue
        row = {"date": work.iloc[i]["date"]}
        all_names = ["const"] + predictor_cols
        for j, name in enumerate(all_names):
            row[f"beta_{name}"] = beta[j]
            row[f"se_{name}"] = se[j]
        results.append(row)
    return pd.DataFrame(results)

## 3k. Granger causality

In [ ]:
def granger_pvalue_table(df, cause_col, effect_col, max_lag=3):
    sample = df[[effect_col, cause_col]].dropna().copy()
    results = grangercausalitytests(sample[[effect_col, cause_col]], maxlag=max_lag, verbose=False)
    rows = []
    for lag in range(1, max_lag + 1):
        p_value = results[lag][0]["ssr_ftest"][1]
        rows.append({"lag": lag, "p_value": round(p_value, 4)})
    out = pd.DataFrame(rows)
    out["cause"] = cause_col
    out["effect"] = effect_col
    return out[["cause", "effect", "lag", "p_value"]]

## 3l. Geweke causality measures

In [ ]:
def geweke_causality(df, var1, var2, lag=2):
    sample = df[[var1, var2]].dropna()
    model = VAR(sample)
    full_results = model.fit(lag)
    residuals = full_results.resid.values
    T = residuals.shape[0]
    Sigma = np.cov(residuals.T)
    Sigma11 = Sigma[0, 0]
    Sigma22 = Sigma[1, 1]
    det_Sigma = np.linalg.det(Sigma)

    Y1 = sample[var1].values
    X1_list = [pd.Series(Y1).shift(i).values for i in range(1, lag + 1)]
    X1 = np.column_stack(X1_list)[lag:]
    Y1_dep = Y1[lag:]
    mask1 = np.all(np.isfinite(X1), axis=1) & np.isfinite(Y1_dep)
    X1, Y1_dep = X1[mask1], Y1_dep[mask1]
    beta1 = np.linalg.inv(X1.T @ X1) @ (X1.T @ Y1_dep)
    Sigma1 = np.var(Y1_dep - X1 @ beta1)

    Y2 = sample[var2].values
    X2_list = [pd.Series(Y2).shift(i).values for i in range(1, lag + 1)]
    X2 = np.column_stack(X2_list)[lag:]
    Y2_dep = Y2[lag:]
    mask2 = np.all(np.isfinite(X2), axis=1) & np.isfinite(Y2_dep)
    X2, Y2_dep = X2[mask2], Y2_dep[mask2]
    beta2 = np.linalg.inv(X2.T @ X2) @ (X2.T @ Y2_dep)
    Sigma2 = np.var(Y2_dep - X2 @ beta2)

    C_21 = np.log(Sigma1 / Sigma11)
    C_12 = np.log(Sigma2 / Sigma22)
    C_inst = np.log((Sigma11 * Sigma22) / det_Sigma)
    C_total = C_21 + C_12 + C_inst

    df_gc = lag
    results_list = [
        {"measure": f"{var2} -> {var1}", "C": C_21, "statistic": T * C_21, "df": df_gc, "p_value": 1.0 - chi2_dist.cdf(max(T * C_21, 0), df_gc)},
        {"measure": f"{var1} -> {var2}", "C": C_12, "statistic": T * C_12, "df": df_gc, "p_value": 1.0 - chi2_dist.cdf(max(T * C_12, 0), df_gc)},
        {"measure": "Instantaneous", "C": C_inst, "statistic": T * C_inst, "df": 1, "p_value": 1.0 - chi2_dist.cdf(max(T * C_inst, 0), 1)},
        {"measure": "Total", "C": C_total, "statistic": T * C_total, "df": 2 * lag + 1, "p_value": 1.0 - chi2_dist.cdf(max(T * C_total, 0), 2 * lag + 1)},
    ]
    out = pd.DataFrame(results_list)
    for col in ["C", "statistic", "p_value"]:
        out[col] = out[col].round(4)
    return out

## 3m. VAR helpers and forecasting

In [ ]:
def choose_var_lag(var_df, maxlags=6):
    model = VAR(var_df)
    lag_selection = model.select_order(maxlags=maxlags)
    lag_table = pd.DataFrame([lag_selection.selected_orders]).rename(index={0: "selected_lag"})
    return lag_selection, lag_table

def fit_var_model(var_df, chosen_lag):
    model = VAR(var_df)
    results = model.fit(chosen_lag)
    return results

In [ ]:
def historical_mean_forecast(train_series):
    return train_series.mean()

In [ ]:
def rolling_forecast_comparison(df, target_col, raw_oil_col, demand_col, supply_col, controls, start_share=0.6):
    work = df.copy()
    work["target_next"] = work[target_col].shift(-1)
    required_cols = ["date", "target_next", target_col, raw_oil_col, demand_col, supply_col] + controls
    work = work[required_cols].dropna().copy()
    start_index = max(int(len(work) * start_share), 36)
    forecasts = []
    for i in range(start_index, len(work)):
        train = work.iloc[:i].copy()
        test = work.iloc[i:i + 1].copy()
        if len(test) == 0:
            continue
        benchmark_forecast = historical_mean_forecast(train["target_next"])
        raw_predictors = [target_col, raw_oil_col] + controls
        X_raw = sm.add_constant(train[raw_predictors])
        raw_model = sm.OLS(train["target_next"], X_raw).fit()
        raw_forecast = raw_model.predict(sm.add_constant(test[raw_predictors], has_constant="add")).iloc[0]
        decomposition_predictors = [target_col, demand_col, supply_col] + controls
        X_decomp = sm.add_constant(train[decomposition_predictors])
        decomp_model = sm.OLS(train["target_next"], X_decomp).fit()
        decomp_forecast = decomp_model.predict(sm.add_constant(test[decomposition_predictors], has_constant="add")).iloc[0]
        forecasts.append({"date": test["date"].iloc[0], "actual": test["target_next"].iloc[0], "benchmark": benchmark_forecast, "raw_oil_model": raw_forecast, "decomposition_model": decomp_forecast})
    forecast_df = pd.DataFrame(forecasts)
    metrics = []
    for model_name in ["benchmark", "raw_oil_model", "decomposition_model"]:
        errors = forecast_df[model_name] - forecast_df["actual"]
        metrics.append({"model": model_name, "rmse": np.sqrt(np.mean(errors ** 2)), "mae": np.mean(np.abs(errors))})
    metrics_df = pd.DataFrame(metrics).round(4)
    return forecast_df, metrics_df

In [ ]:
def split_sample(df, split_date):
    early = df[df["date"] < split_date].copy()
    late = df[df["date"] >= split_date].copy()
    return early, late

print("All functions defined.")

# 4. Data loading

In [ ]:
DATA_PATH = Path("../data/raw/data_hec_projet_1.xlsx")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/data_hec_projet_1.xlsx")
print("Excel file found:", DATA_PATH.exists())

In [ ]:
daily_raw = load_data_sheet(DATA_PATH, "Daily", DAILY_COLUMNS)
monthly_raw = load_data_sheet(DATA_PATH, "Monthly", MONTHLY_COLUMNS)

print("Daily shape:", daily_raw.shape, "| Range:", daily_raw["date"].min().date(), "to", daily_raw["date"].max().date())
print("Monthly shape:", monthly_raw.shape, "| Range:", monthly_raw["date"].min().date(), "to", monthly_raw["date"].max().date())

# 5. Data cleaning and monthly aggregation

In [ ]:
monthly_market_data = aggregate_daily_to_monthly(daily_raw)
monthly_merged = (
    monthly_market_data.merge(monthly_raw, on="date", how="inner")
    .sort_values("date")
    .reset_index(drop=True)
)
print("Merged monthly shape:", monthly_merged.shape)

# 6. Variable construction

In [ ]:
project_df = build_project_variables(monthly_merged)

key_columns = [
    "date", "wti_return", "sp500_return", "hy_change",
    "term_spread", "gold_return", "sp500_realized_vol", "cfnai", "ism_mfg",
]
print("Project dataset shape:", project_df.shape)
display(project_df[key_columns].head(10))

# 7. Descriptive statistics and diagnostics

We combine several diagnostic tools:
- Summary statistics (moments 1–4),
- ADF tests for stationarity,
- Ljung-Box tests for autocorrelation,
- Jarque-Bera tests for normality,
- ACF of squared returns for volatility clustering.

In [ ]:
descriptive_columns = [
    "wti_return", "sp500_return", "hy_change", "term_spread",
    "gold_return", "sp500_realized_vol", "cfnai", "ism_mfg",
]

summary_table = project_df[descriptive_columns].describe().T
summary_table["skew"] = project_df[descriptive_columns].skew()
summary_table["kurtosis"] = project_df[descriptive_columns].kurtosis()
summary_table = summary_table[["count", "mean", "std", "min", "25%", "50%", "75%", "max", "skew", "kurtosis"]].round(4)

print("Table 1. Summary statistics")
display(summary_table)

In [ ]:
print("Table 2. Correlation matrix")
display(project_df[descriptive_columns].corr().round(3))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, (col, title) in zip(axes, [
    ("wti_return", "Figure 1. Monthly WTI return"),
    ("sp500_return", "Figure 2. Monthly S&P 500 return"),
    ("hy_change", "Figure 3. Monthly HY spread change"),
]):
    ax.plot(project_df["date"], project_df[col])
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
stationarity_columns = ["wti_return", "sp500_return", "hy_change", "term_spread", "gold_return", "cfnai", "ism_mfg"]

print("Table 3a. ADF stationarity tests")
display(run_adf_table(project_df, stationarity_columns))

In [ ]:
print("Table 3b. Ljung-Box autocorrelation and Jarque-Bera normality tests")
display(run_ljungbox_jb_table(project_df, stationarity_columns))

### Volatility clustering (ACF of squared returns)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["wti_return", "sp500_return", "gold_return"]):
    s = project_df[col].dropna().values
    acf_vals = acf(s**2, nlags=12, fft=True)
    T = len(s)
    ax.bar(range(1, 13), acf_vals[1:13], color="darkorange", width=0.6, alpha=0.8)
    ax.axhline(y=1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8)
    ax.axhline(y=-1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8)
    ax.axhline(y=0, color="black", linewidth=0.5)
    ax.set_title(f"ACF of {col} squared")
    ax.set_xlabel("Lag")
plt.suptitle("Figure 4. Volatility clustering", fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
for col in ["wti_return", "sp500_return", "gold_return"]:
    lb = acorr_ljungbox(project_df[col].dropna()**2, lags=[6, 12], return_df=True)
    print(f"{col} squared: LB(6) p={lb.loc[6,'lb_pvalue']:.4f}, LB(12) p={lb.loc[12,'lb_pvalue']:.4f}")

# 8. Oil shock decomposition

We decompose monthly WTI returns using two approaches:
- **Approach A**: OLS regression on CFNAI. Fitted = demand component, residual = supply component.
- **Approach B**: ISM regime dummy (expansion if ISM > 50, contraction otherwise).

In [ ]:
decomposition_df, scratch_res = decompose_oil_returns_scratch(
    project_df, oil_return_col="wti_return", activity_col="cfnai", prefix="baseline"
)

print("Table 4. Oil decomposition: r_wti = alpha + beta * CFNAI + epsilon")
display(ols_summary_df(scratch_res))
print(f"R squared = {scratch_res['R2']:.4f}, T = {scratch_res['T']}")

In [ ]:
print("Residual diagnostics:")
display(run_diagnostic_residuals(scratch_res["resid"]))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
acf_vals = acf(scratch_res["resid"], nlags=12, fft=True)
T = len(scratch_res["resid"])
ax.bar(range(1, 13), acf_vals[1:13], color="steelblue", width=0.6, alpha=0.8)
ax.axhline(y=1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8)
ax.axhline(y=-1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_title("Figure 5. ACF of decomposition residuals (OLS)")
ax.set_xlabel("Lag")
ax.set_ylabel("ACF")
plt.tight_layout()
plt.show()

## 8b. MA(1) correction

The residuals show significant autocorrelation at lag 6. We estimate the same decomposition with MA(1) errors using maximum likelihood with recursive innovations and OPG standard errors.

In [ ]:
sample = project_df[["wti_return", "cfnai"]].dropna()
ma1_res = estimate_ma1(sample["wti_return"].values, sample["cfnai"].values.reshape(-1, 1), ["cfnai"])

print("Table 5. OLS vs MA(1) MLE comparison")
display(ma1_comparison_table(ma1_res))

In [ ]:
print(f"MA(1) theta = {ma1_res['theta']:.4f}, OPG SE = {ma1_res['se'][-1]:.4f}")
print(f"Sigma OLS = {np.sqrt(scratch_res['sigma2']):.4f}, Sigma MLE = {ma1_res['sigma']:.4f}")
print("Beta CFNAI moves from {:.4f} (OLS) to {:.4f} (MLE) -- decomposition is robust.".format(scratch_res["beta"][1], ma1_res["beta"][1]))

In [ ]:
print("Residual diagnostics after MA(1) correction:")
display(run_diagnostic_residuals(ma1_res["innovations"]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
nlags = 12
acf_ols = acf(scratch_res["resid"], nlags=nlags, fft=True)
acf_ma1 = acf(ma1_res["innovations"], nlags=nlags, fft=True)
lags = np.arange(1, nlags + 1)
T = len(scratch_res["resid"])

ax.stem(lags - 0.15, acf_ols[1:], linefmt="C0-", markerfmt="C0o", basefmt=" ", label="OLS residuals")
ax.stem(lags + 0.15, acf_ma1[1:], linefmt="C1-", markerfmt="C1s", basefmt=" ", label="MA(1) innovations")
ax.axhline(y=1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axhline(y=-1.96/np.sqrt(T), color="red", linestyle="--", linewidth=0.8, alpha=0.6)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_title("Figure 6. Residual ACF: OLS vs MA(1)")
ax.set_xlabel("Lag")
ax.set_ylabel("Autocorrelation")
ax.legend()
ax.set_xticks(lags)
plt.tight_layout()
plt.show()

In [ ]:
decomposition_df = add_regime_variables(decomposition_df, oil_col="wti_return", ism_col="ism_mfg")

n_exp = decomposition_df["expansion"].sum()
n_con = decomposition_df["contraction"].sum()
print(f"Expansion months (ISM > 50): {n_exp} ({100*n_exp/(n_exp+n_con):.1f}%)")
print(f"Contraction months (ISM <= 50): {n_con} ({100*n_con/(n_exp+n_con):.1f}%)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(decomposition_df["date"], decomposition_df["wti_return"], label="WTI return", alpha=0.8)
axes[0].plot(decomposition_df["date"], decomposition_df["baseline_oil_demand_component"], label="Demand component", linewidth=1.5)
axes[0].set_title("Figure 7. WTI return and demand-related component")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(decomposition_df["date"], decomposition_df["baseline_oil_supply_component"], label="Supply/residual component", color="firebrick")
axes[1].set_title("Figure 8. Residual oil component (supply/non-demand)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 9. Predictive regressions

We estimate one-month-ahead regressions for S&P 500 returns and HY spread changes.

In [ ]:
target_map = {
    "sp500_return": "S&P 500 next-month return",
    "hy_change": "HY next-month change",
}

decomposition_models = {}

for target, label in target_map.items():
    decomp_model, decomp_sample = fit_predictive_regression(
        decomposition_df, dependent_col=target,
        predictor_cols=["baseline_oil_demand_component", "baseline_oil_supply_component", target, "term_spread", "gold_return", "sp500_realized_vol"],
        horizon=1, cov_type="HC1",
    )
    decomposition_models[target] = decomp_model
    print(f"Decomposition model: {label} (N = {len(decomp_sample)})")
    display(regression_results_table(decomp_model))
    for line in interpret_two_component_model(decomp_model, "baseline_oil_demand_component", "baseline_oil_supply_component"):
        print(f"  {line}")
    print("Residual diagnostics:")
    display(run_diagnostic_residuals(decomp_model.resid))
    print()

In [ ]:
for target, label in target_map.items():
    regime_model, regime_sample = fit_predictive_regression(
        decomposition_df, dependent_col=target,
        predictor_cols=["oil_expansion", "oil_contraction", target, "term_spread", "gold_return", "sp500_realized_vol"],
        horizon=1, cov_type="HC1",
    )
    print(f"Regime model: {label} (N = {len(regime_sample)})")
    display(regression_results_table(regime_model))
    print("Residual diagnostics:")
    display(run_diagnostic_residuals(regime_model.resid))
    print()

## 9b. Stability of predictive coefficients

We estimate the decomposition predictive regression on a rolling 60-month window.

In [ ]:
rolling_df = rolling_predictive_coefficients(
    decomposition_df, dependent_col="sp500_return",
    predictor_cols=["baseline_oil_demand_component", "baseline_oil_supply_component", "sp500_return", "term_spread", "gold_return", "sp500_realized_vol"],
    window=60, horizon=1,
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, comp, color, title in [
    (axes[0], "baseline_oil_demand_component", "steelblue", "Demand component"),
    (axes[1], "baseline_oil_supply_component", "firebrick", "Supply component"),
]:
    beta_col = f"beta_{comp}"
    se_col = f"se_{comp}"
    ax.plot(rolling_df["date"], rolling_df[beta_col], color=color, linewidth=1.2)
    ax.fill_between(rolling_df["date"], rolling_df[beta_col] - 2 * rolling_df[se_col], rolling_df[beta_col] + 2 * rolling_df[se_col], alpha=0.15, color=color)
    ax.axhline(y=0, color="black", linewidth=0.5, linestyle="--")
    ax.set_title(f"Figure 9. Rolling beta: {title} (60-month window)")
    ax.set_ylabel("Coefficient")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 10. Causality tests

We use two approaches:
- **Granger F-tests**,
- **Geweke (1982) causality measures** with chi-squared tests.

In [ ]:
granger_specs = [
    ("baseline_oil_demand_component", "sp500_return"),
    ("baseline_oil_supply_component", "sp500_return"),
    ("baseline_oil_demand_component", "hy_change"),
    ("baseline_oil_supply_component", "hy_change"),
]
granger_tables = [granger_pvalue_table(decomposition_df, c, e, max_lag=3) for c, e in granger_specs]
print("Table 6. Granger causality p-values")
display(pd.concat(granger_tables, ignore_index=True))

In [ ]:
daily_returns = daily_raw[["date", "sp500", "wti"]].copy()
daily_returns["r_sp500"] = np.log(daily_returns["sp500"]).diff()
daily_returns["r_wti"] = np.log(daily_returns["wti"]).diff()
daily_returns = daily_returns.dropna()

var_sp = daily_returns["r_sp500"].rolling(60).var()
var_wti = daily_returns["r_wti"].rolling(60).var()
var_df = pd.DataFrame({"sp500_var": var_sp, "wti_var": var_wti}).dropna()

print("Table 7. Geweke causality measures")
display(geweke_causality(var_df, "wti_var", "sp500_var", lag=2))

In [ ]:
var_df_dated = var_df.copy()
var_df_dated["year"] = daily_returns.loc[var_df.index, "date"].dt.year.values

blocks = []
for start in range(2000, 2025, 5):
    sub = var_df_dated[(var_df_dated["year"] >= start) & (var_df_dated["year"] < start + 5)][["wti_var", "sp500_var"]]
    if len(sub) < 200:
        continue
    g = geweke_causality(sub, "wti_var", "sp500_var", lag=2)
    wti_to_sp = g[g["measure"].str.contains("wti_var -> sp500_var")]
    if len(wti_to_sp) > 0:
        blocks.append({"period": f"{start}-{start+4}", "C": wti_to_sp["C"].values[0], "statistic": wti_to_sp["statistic"].values[0], "p_value": wti_to_sp["p_value"].values[0]})

print("Table 8. Geweke WTI -> S&P 500 by sub-period")
display(pd.DataFrame(blocks))

# 11. VAR and impulse responses

Reduced-form VAR: lag selection, stability check, residual diagnostics, orthogonalized IRFs, and variance decomposition.

In [ ]:
var_columns = ["wti_return", "sp500_return", "hy_change", "term_spread", "gold_return", "ism_mfg"]
var_df = decomposition_df[["date"] + var_columns].dropna().set_index("date")

var_model = VAR(var_df)
lag_selection = var_model.select_order(maxlags=6)
lag_choice = lag_selection.selected_orders["bic"]
if lag_choice == 0:
    lag_choice = max(1, lag_selection.selected_orders["aic"])

print("Table 9. VAR lag selection")
display(pd.DataFrame([lag_selection.selected_orders], index=["selected_lag"]))
print("Chosen lag:", lag_choice)

In [ ]:
var_results = var_model.fit(lag_choice)

print(f"VAR is stable: {var_results.is_stable()}")

print("\nTable 10. Ljung-Box on VAR residuals (lag 12)")
for col in var_columns:
    lb = acorr_ljungbox(var_results.resid[col], lags=[12], return_df=True)
    print(f"  {col}: LB(12) stat={lb.loc[12,'lb_stat']:.2f}, p={lb.loc[12,'lb_pvalue']:.4f}")

In [ ]:
irf = var_results.irf(24)
irf_err = irf.errband_mc(orth=True, repl=300, signif=0.05, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, response, title in [
    (axes[0], "sp500_return", "Figure 10. IRF: S&P 500 response to oil shock"),
    (axes[1], "hy_change", "Figure 11. IRF: HY spread response to oil shock"),
]:
    impulse_idx = var_columns.index("wti_return")
    response_idx = var_columns.index(response)
    irf_vals = irf.orth_irfs[:, response_idx, impulse_idx]
    lower = irf_err[0][:, response_idx, impulse_idx]
    upper = irf_err[1][:, response_idx, impulse_idx]
    ax.plot(range(25), irf_vals, color="navy", linewidth=1.5)
    ax.fill_between(range(25), lower, upper, alpha=0.15, color="steelblue")
    ax.axhline(y=0, color="black", linewidth=0.5, linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Months")
    ax.set_ylabel("Response")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fevd = var_results.fevd(12)
print("Table 11. Forecast error variance decomposition at horizon 12")
fevd_table = pd.DataFrame(fevd.decomp[11], index=var_columns, columns=var_columns).round(4)
display(fevd_table[["wti_return"]].rename(columns={"wti_return": "share_explained_by_oil"}))

# 12. Out-of-sample forecasting

Historical mean benchmark, raw oil model, and decomposition model. Plus a VAR forecast exercise.

In [ ]:
for target, label in target_map.items():
    _, metrics_df = rolling_forecast_comparison(
        decomposition_df, target_col=target, raw_oil_col="wti_return",
        demand_col="baseline_oil_demand_component", supply_col="baseline_oil_supply_component",
        controls=["term_spread", "gold_return", "sp500_realized_vol"], start_share=0.6,
    )
    print(f"Table 12. Forecast comparison: {label}")
    display(metrics_df)
    print()

In [ ]:
split_date = "2020-01-01"
train_var = var_df[var_df.index < split_date]
test_var = var_df[var_df.index >= split_date]

if len(test_var) > 0 and len(train_var) > lag_choice:
    var_train_model = VAR(train_var).fit(lag_choice)
    n_forecast = min(len(test_var), 12)
    forecast_vals = var_train_model.forecast(train_var.values[-lag_choice:], steps=n_forecast)
    forecast_df = pd.DataFrame(forecast_vals, index=test_var.index[:n_forecast], columns=var_columns)

    print("Table 13. VAR forecast RMSE (train < 2020, test >= 2020)")
    for col in ["sp500_return", "hy_change"]:
        actual = test_var[col].iloc[:n_forecast]
        pred = forecast_df[col]
        rmse = np.sqrt(np.mean((actual - pred)**2))
        print(f"  {col}: RMSE = {rmse:.4f}")

In [ ]:
if len(test_var) > 0 and len(train_var) > lag_choice:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, col, title in [
        (axes[0], "sp500_return", "Figure 12. VAR forecast vs realized: S&P 500"),
        (axes[1], "hy_change", "Figure 13. VAR forecast vs realized: HY change"),
    ]:
        ax.plot(test_var.index[:n_forecast], test_var[col].iloc[:n_forecast], label="Realized", marker="o", markersize=4)
        ax.plot(forecast_df.index, forecast_df[col], label="VAR forecast", marker="s", markersize=4)
        ax.set_title(title)
        ax.legend()
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 13. Robustness checks

1. Brent instead of WTI,
2. MSCI EM instead of S&P 500,
3. Sub-period analysis.

In [ ]:
brent_df, _ = decompose_oil_returns(decomposition_df, oil_return_col="brent_return", activity_col="cfnai", prefix="brent")
brent_model, brent_sample = fit_predictive_regression(
    brent_df, dependent_col="sp500_return",
    predictor_cols=["brent_oil_demand_component", "brent_oil_supply_component", "sp500_return", "term_spread", "gold_return", "sp500_realized_vol"],
    horizon=1, cov_type="HC1",
)

em_model, em_sample = fit_predictive_regression(
    decomposition_df, dependent_col="msci_em_return",
    predictor_cols=["baseline_oil_demand_component", "baseline_oil_supply_component", "msci_em_return", "term_spread", "gold_return", "sp500_realized_vol"],
    horizon=1, cov_type="HC1",
)

robustness_rows = []
for name, model, demand_name, supply_name in [
    ("Brent instead of WTI", brent_model, "brent_oil_demand_component", "brent_oil_supply_component"),
    ("MSCI EM instead of S&P", em_model, "baseline_oil_demand_component", "baseline_oil_supply_component"),
]:
    robustness_rows.append({"check": name, "demand_coef": model.params.get(demand_name, np.nan), "demand_pval": model.pvalues.get(demand_name, np.nan), "supply_coef": model.params.get(supply_name, np.nan), "supply_pval": model.pvalues.get(supply_name, np.nan)})

print("Table 14. Robustness")
display(pd.DataFrame(robustness_rows).round(4))

In [ ]:
sub_periods = [
    ("1990-2007", "1990-01-01", "2008-01-01"),
    ("2008-2014", "2008-01-01", "2015-01-01"),
    ("2015-2026", "2015-01-01", "2027-01-01"),
]

sub_rows = []
for label, start, end in sub_periods:
    sub = decomposition_df[(decomposition_df["date"] >= start) & (decomposition_df["date"] < end)]
    if len(sub) < 30:
        continue
    sub_model, sub_sample = fit_predictive_regression(
        sub, dependent_col="sp500_return",
        predictor_cols=["baseline_oil_demand_component", "baseline_oil_supply_component", "sp500_return", "term_spread", "gold_return", "sp500_realized_vol"],
        horizon=1, cov_type="HC1",
    )
    sub_rows.append({"period": label, "N": len(sub_sample), "demand_coef": sub_model.params.get("baseline_oil_demand_component", np.nan), "demand_pval": sub_model.pvalues.get("baseline_oil_demand_component", np.nan), "supply_coef": sub_model.params.get("baseline_oil_supply_component", np.nan), "supply_pval": sub_model.pvalues.get("baseline_oil_supply_component", np.nan)})

print("Table 15. Sub-period analysis")
display(pd.DataFrame(sub_rows).round(4))

# 14. Main conclusions

The main message of the notebook should be read in a reduced-form sense. The fitted oil component is interpreted as demand-related, while the residual component is interpreted as non-demand-related. The project provides a transparent and coherent empirical framework rather than a full structural identification strategy.

In [ ]:
print("Main takeaway for S&P 500 returns")
for line in interpret_two_component_model(decomposition_models["sp500_return"], "baseline_oil_demand_component", "baseline_oil_supply_component"):
    print(f"  {line}")

print("\nMain takeaway for HY changes")
for line in interpret_two_component_model(decomposition_models["hy_change"], "baseline_oil_demand_component", "baseline_oil_supply_component"):
    print(f"  {line}")